In [1]:
%cd ..

from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.magic("%load_ext autoreload")
    ipython.magic("%autoreload 2")

import torch

import torch as t
from torch import Tensor
import einops

from transformer_lens import HookedTransformer

from eap.eap_wrapper import EAP

from jaxtyping import Float

device = t.device('cuda') if t.cuda.is_available() else t.device('cpu')
print(f'Device: {device}')

/Users/molotkova_s/Documents/mlteam/edge-attribution-patching


/var/folders/nd/bb2tsw517s32_mw46qlrq2v40000gn/T/ipykernel_20354/2089673066.py:6: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("%load_ext autoreload")
/var/folders/nd/bb2tsw517s32_mw46qlrq2v40000gn/T/ipykernel_20354/2089673066.py:7: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("%autoreload 2")


Device: cpu


# Model Setup

In [2]:
model = HookedTransformer.from_pretrained(
    'gpt2-small',
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device=device,
)
model.set_use_hook_mlp_in(True)
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)

Using pad_token, but it is not set yet.


Loaded pretrained model gpt2-small into HookedTransformer


# Dataset Setup

In [4]:
from ioi_dataset import IOIDataset, format_prompt, make_table
N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

make_table(
  colnames = ["IOI prompt", "IOI subj", "IOI indirect obj", "ABC prompt"],
  cols = [
    map(format_prompt, clean_dataset.sentences),
    model.to_string(clean_dataset.s_tokenIDs).split(),
    model.to_string(clean_dataset.io_tokenIDs).split(),
    map(format_prompt, corr_dataset.sentences),
  ],
  title = "Sentences from IOI vs ABC distribution",
)

                                      Sentences from IOI vs ABC distribution                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ IOI prompt                              ┃ IOI subj ┃ IOI indirect obj ┃ ABC prompt                              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ When Victoria and Jane got a snack at   │ Jane     │ Victoria         │ When Jake and George got a snack at the │
│ the store, Jane decided to give it to   │          │                  │ store, River decided to give it to      │
│ Victoria                                │          │                  │ Victoria                                │
│                                         │          │                  │                                         │
│ When Sullivan and Rose got a necklace   │ Sullivan │ Rose             │ When Roman and Alan got a necklace at   │
│ at the garden, Sullivan decided to give │          │                  │ the garden, Connor decided to give it   │
│ it to Rose                              │          │                  │ to Rose                                 │
│                                         │          │                  │                                         │
│ When Alan and Alex got a drink at the   │ Alex     │ Alan             │ When Jack and Jacob got a drink at the  │
│ store, Alex decided to give it to Alan  │          │                  │ store, Collins decided to give it to    │
│                                         │          │                  │ Alan                                    │
│                                         │          │                  │                                         │
│ Then, Jessica and Crystal had a long    │ Jessica  │ Crystal          │ Then, Patrick and Kevin had a long      │
│ argument, and afterwards Jessica said   │          │                  │ argument, and afterwards Roman said to  │
│ to Crystal                              │          │                  │ Crystal                                 │
│                                         │          │                  │                                         │
│ Then, Jonathan and Kevin were working   │ Kevin    │ Jonathan         │ Then, Joshua and Max were working at    │
│ at the school. Kevin decided to give a  │          │                  │ the school. Steven decided to give a    │
│ necklace to Jonathan                    │          │                  │ necklace to Jonathan                    │
│                                         │          │                  │                                         │
└─────────────────────────────────────────┴──────────┴──────────────────┴─────────────────────────────────────────┘

# Metric Setup

In [5]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()

with t.no_grad():
    clean_logits = model(clean_dataset.toks)
    corrupt_logits = model(corr_dataset.toks)
    clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
    corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

Clean direction: 2.8051624298095703, Corrupt direction: 1.7837640047073364
Clean metric: 1.0, Corrupt metric: 0.0


# Run Experiment

In [12]:
model.reset_hooks()

graph = EAP(
    model,
    clean_dataset.toks,
    corr_dataset.toks,
    ioi_metric,
    upstream_nodes=["mlp", "head"],
    downstream_nodes=["mlp", "head"],
    batch_size=25
)

top_edges = graph.top_edges(n=100, abs_scores=True)
for from_edge, to_edge, score in top_edges:
    print(f'{from_edge} -> [{round(score, 3)}] -> {to_edge}')

Saving activations requires 0.0004 GB of memory per token


100%|██████████| 1/1 [00:03<00:00,  3.62s/it]

head.9.9 -> [-0.029] -> head.11.10.q
head.10.7 -> [0.028] -> head.11.10.q
head.5.5 -> [0.024] -> head.8.6.v
mlp.0 -> [-0.023] -> mlp.4
head.5.5 -> [-0.021] -> mlp.5
head.9.9 -> [-0.02] -> head.10.7.q
mlp.0 -> [0.02] -> head.6.9.q
head.5.5 -> [-0.016] -> head.6.9.q
mlp.0 -> [-0.016] -> mlp.5
head.3.0 -> [-0.015] -> mlp.5
head.9.6 -> [-0.014] -> head.11.10.q
mlp.5 -> [-0.013] -> mlp.6
head.4.11 -> [0.013] -> head.6.9.k
mlp.0 -> [-0.013] -> head.11.10.k
head.3.0 -> [-0.013] -> mlp.4
head.4.11 -> [-0.012] -> mlp.5
head.9.6 -> [-0.012] -> head.10.7.q
head.5.5 -> [-0.012] -> mlp.6
mlp.0 -> [-0.012] -> mlp.3
mlp.0 -> [-0.011] -> head.10.7.k
head.5.5 -> [0.01] -> head.7.9.v
mlp.1 -> [-0.01] -> mlp.4
mlp.0 -> [0.009] -> head.3.0.k
head.10.10 -> [-0.009] -> head.11.10.q
head.4.11 -> [-0.009] -> mlp.4
head.5.5 -> [0.009] -> head.8.10.v
head.9.6 -> [-0.009] -> head.10.0.q
mlp.0 -> [0.008] -> head.3.0.q
mlp.0 -> [-0.008] -> head.10.7.v
head.6.9 -> [0.008] -> head.8.6.v
head.8.6 -> [0.008] -> head.9

In [13]:
graph.show(threshold=0.004)

Saving graph


<AGraph b'root' <Swig Object of type 'Agraph_t *' at 0x13c414ae0>>